# Mackey–Glass: a fixed-parameter periodic-orbit theorem

This notebook documents and checks an executed computer-assisted proof for
\[
x'(t)=0.2x(t-17)/(1+x(t-17)^{10})-0.1x(t).
\]
All four physical parameters are fixed. Frequency is the unknown scalar. The **original surrogate waveform is validated with zero true-equation Newton corrections**.

The source candidate remains in `outputs_zero_correction/MG_zero_correction_candidate.mat`. The proof, certificate, and audit are in `CAP/Mackey-Glass_PO/`. This replaces the earlier free-damping validation claim; it does not use that driver's success flag or reported radius.

By default, Run All checks and displays the executed certificate. Set `MG_RUN_FIXED_PARAMETER_PROOF=1` before running to regenerate the dyadic candidate data and rerun MATLAB with INTLAB 13. NumPy and SciPy are required for preparation. MATLAB and INTLAB are needed only for recomputation.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib
import numpy as np
from scipy.io import loadmat

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "CAP/Mackey-Glass_PO/proof_MG_fixed_parameter.m").is_file())
PROOF_DIR = REPO_ROOT / "CAP/Mackey-Glass_PO"
PROOF_OUTPUT = PROOF_DIR / "outputs_fixed_parameter"
RUN_INTERVAL_PROOF = os.environ.get("MG_RUN_FIXED_PARAMETER_PROOF", "0") == "1"
print("Proof directory:", PROOF_DIR)
print("Mode:", "regenerate interval certificate" if RUN_INTERVAL_PROOF else "check executed certificate")


## 1. Unchanged surrogate input

The source export has degree 40 and is stored in a zero-padded degree-200 array. The proof uses an 80-mode inverse core. Removing exact zero padding does not alter the waveform. We check the source hash and exact equality of its nonzero Fourier coefficients and frequency.


In [ ]:
diagnostics = json.loads((PROOF_OUTPUT / "candidate_diagnostics.json").read_text())
source = REPO_ROOT / "notebooks/mackey_glass/outputs_zero_correction/MG_zero_correction_candidate.mat"
assert hashlib.sha256(source.read_bytes()).hexdigest() == diagnostics["source_sha256"]
data = loadmat(PROOF_OUTPUT / "mg_fixed_parameter_candidate.mat")
np.testing.assert_array_equal(data["u"], data["initial_u"])
assert data["omega"].item() == data["initial_omega"].item()
assert data["newton_steps"].item() == 0
assert diagnostics["source_truncation_weighted_l1"] == 0
print(json.dumps(diagnostics, indent=2))


## 2. Fixed-parameter Fourier formulation

Set $s=t/10$, $\tau=17/10$, $\theta=\omega s$, and $G(z)=z/(1+z^{10})$. The equation becomes
\[
\omega w_\theta+w-2G(w(\theta-\omega\tau))=0.
\]
The proof norm is
\[
|\delta\omega|+\sum_{k\in\mathbb Z}|\delta u_k|1.01^{|k|}.
\]
A finite linear phase condition removes time translation. The integrated residual divides nonzero Fourier equations by $ik\omega_{\rm ref}$ and retains the zero equation separately. The derivative with respect to frequency includes the derivative of the delayed phase $\omega\tau$.

For each exact pole $p_j=e^{i\pi(2j-1)/10}$, degree-160 approximate reciprocals are validated by interval convolution and a Banach-algebra Neumann-series bound. The same reciprocal estimates exclude poles and control $G'$ and $G''$ throughout the **entire** contraction ball. The finite matrix includes inputs through mode 240 and outputs through mode 400; an analytic bound covers every remaining input mode to infinity.

See `CAP/Mackey-Glass_PO/FIXED_PARAMETER_PROOF.md` for the operators, all finite/tail estimates, and derivation of the frequency-variation bounds.


In [ ]:
if RUN_INTERVAL_PROOF:
    subprocess.run([sys.executable, str(PROOF_DIR / "fixed_parameter_candidate.py"),
                    "--newton-steps", "0"], check=True)
    matlab = os.environ.get("MATLAB_BINARY", "matlab")
    intlab = os.environ.get("INTLAB_DIR")
    if not intlab:
        raise RuntimeError("Set INTLAB_DIR to your INTLAB installation directory")
    quote = lambda p: str(p).replace("'", "''")
    command = (f"addpath('{quote(intlab)}'); startintlab; "
               f"run('{quote(PROOF_DIR / 'proof_MG_fixed_parameter.m')}');")
    result = subprocess.run([matlab, "-batch", command,
                             "-logfile", str(PROOF_OUTPUT / "cap.log")],
                            capture_output=True, text=True, timeout=600)
    print(result.stdout[-6000:])
    if result.returncode:
        print(result.stderr)
        raise RuntimeError("Fixed-parameter interval proof failed.")
else:
    print("Using the executed MATLAB/INTLAB certificate; no proof rerun in this notebook session.")
certificate = json.loads((PROOF_OUTPUT / "mg_fixed_parameter_certificate.json").read_text())
assert certificate["proof_success"] and certificate["fixed_parameters"]
assert certificate["complete_infinite_tail"]
assert certificate["true_system_newton_steps"] == 0
assert certificate["radii_polynomial_upper"] < 0
assert certificate["contraction_upper"] < 1
assert certificate["pole_neighborhood_factor"] < 1
assert certificate["nonconstant_coefficient_lower"] > 0
print(json.dumps(certificate, indent=2))


## 3. The established theorem

There is a nonstationary periodic solution of the fixed physical equation within **0.0006** of the original surrogate candidate in the augmented norm above. The solution is unique in the stated ball and finite phase slice. Its physical period lies in the interval printed below. This proves periodic-orbit existence; it does not assert orbital stability, chaos, or that the enclosed period is minimal.


In [ ]:
print(f"Certified augmented radius: {certificate['radius_decimal']}")
print(f"Physical period: [{certificate['period_lower']:.14f}, {certificate['period_upper']:.14f}]")
print(f"Contraction bound: {certificate['contraction_upper']:.12f}")
print(f"True-system Newton corrections: {certificate['true_system_newton_steps']}")
tests = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(PROOF_DIR),
                        "-p", "test_fixed_parameter.py", "-v"], capture_output=True, text=True)
print(tests.stdout + tests.stderr)
assert tests.returncode == 0


## 4. Verification scope

The certificate was generated by MATLAB R2024b with INTLAB 13. All proof convolutions, finite matrix products, weighted norms, and final inequalities use outward-rounded intervals. Floating-point preparation is not itself used as a proof.

The Python checks independently cross-check the frequency/state Jacobian, partial fractions, unchanged source coefficients, and certificate consistency. Executed interval guard tests additionally reject a radius of `1e-12` and a pole-invalid radius of `0.2`, leaving no success certificate. Their record is `outputs_fixed_parameter/guard_tests.log`.

The older free-damping driver and its outputs remain historical artifacts. The new theorem and manuscript use only the fixed-parameter certificate.
